In [15]:
import cv2
import pywt
import numpy as np
from skimage.feature import hog
from keras.models import load_model
import joblib

In [16]:
cnn=load_model("cnn_for_identification.h5")
rf=joblib.load("rf_model.pkl")
svm=joblib.load("svm_model.pkl")
knn=joblib.load("knn_model.pkl")
LogR=joblib.load("logr_model.pkl")

#GUI

In [17]:
import customtkinter as ctk
from customtkinter import CTkImage
from tkinter import filedialog, Toplevel
from PIL import Image
import numpy as np


# ------------------------------
# Global variables
# ------------------------------
gallery_items = []
active_options = None
active_owner = None
animation_speed = 15  # pixels per frame


# ------------------------------
# Functions
# ------------------------------
def select_images():
    file_paths = filedialog.askopenfilenames(
        filetypes=[("Image files", "*.jpg;*.jpeg;*.png;*.bmp")]
    )
    for file_path in file_paths:
        add_to_gallery(file_path)
    arrange_gallery()
    pred_text.set("Prediction: None")  # Reset prediction on new selection


def add_to_gallery(file_path):
    global active_options, active_owner

    card = ctk.CTkFrame(gallery_frame, corner_radius=15, fg_color="#252525")
    thumb_size = screen_width // 12
    img = Image.open(file_path).resize((thumb_size, thumb_size))
    img_tk = CTkImage(light_image=img, size=(thumb_size, thumb_size))  # ✅ fixed

    # Store original image and path
    card.img = Image.open(file_path)
    card.file_path = file_path

    img_container = ctk.CTkFrame(card, width=thumb_size, height=thumb_size,
                                 corner_radius=15, fg_color="#1a1a1a")
    img_container.pack_propagate(False)
    img_container.pack(padx=5, pady=5)

    img_label = ctk.CTkLabel(img_container, image=img_tk, text="")
    img_label.image = img_tk
    img_label.pack(expand=True, fill="both")

    # Options frame (hidden initially)
    options = ctk.CTkFrame(img_container, fg_color="black", corner_radius=10)
    options.place(relx=0.5, y=-100, anchor="n")
    options.place_forget()

    btn_zoom = ctk.CTkButton(options, text="🔍 Zoom",
                             command=lambda: open_zoom_window(file_path),
                             width=90, height=28, fg_color="#00c6ff", hover_color="#0072ff")
    btn_zoom.pack(padx=5, pady=2)

    btn_remove = ctk.CTkButton(options, text="🗑 Remove",
                               command=lambda: remove_from_gallery(card),
                               width=90, height=28, fg_color="#ff3b30", hover_color="#cc2e25")
    btn_remove.pack(padx=5, pady=2)

    # Click to toggle options
    def on_click(event):
        global active_options, active_owner
        if active_options and active_options.winfo_exists() and active_options != options:
            slide_hide(active_options)
        if options.winfo_ismapped():
            slide_hide(options)
            active_options = None
            active_owner = None
        else:
            options.place(relx=0.5, y=-100, anchor="n")
            options.update()
            options.lift()
            slide_show(options, 5)
            active_options = options
            active_owner = img_container

    img_label.bind("<Button-1>", on_click)

    # Hover effect
    def on_enter(e):
        img_container.configure(fg_color="#2e2e2e")
        card.configure(fg_color="#303030")

    def on_exit(e):
        img_container.configure(fg_color="#1a1a1a")
        card.configure(fg_color="#252525")

    img_container.bind("<Enter>", on_enter)
    img_container.bind("<Leave>", on_exit)

    gallery_items.append(card)


def slide_show(frame, step=5):
    x, y = frame.winfo_x(), frame.winfo_y()
    target_y = frame.master.winfo_height() // 2 - frame.winfo_height() // 2
    if y < target_y:
        y = min(y + step, target_y)
        frame.place_configure(y=y)
        frame.after(10, lambda: slide_show(frame, step))


def slide_hide(frame, step=5):
    if not frame.winfo_exists():
        return
    x, y = frame.winfo_x(), frame.winfo_y()
    if y > -frame.winfo_height():
        y = max(y - step, -frame.winfo_height())
        frame.place_configure(y=y)
        frame.after(10, lambda: slide_hide(frame, step))
    else:
        frame.place_forget()


def remove_from_gallery(frame):
    global active_options, active_owner
    if active_owner == frame:
        if active_options and active_options.winfo_exists():
            active_options.place_forget()
        active_options = None
        active_owner = None
    frame.destroy()
    if frame in gallery_items:
        gallery_items.remove(frame)
    arrange_gallery()


def remove_all_images():
    global active_options, active_owner
    active_options = None
    active_owner = None
    for frame in gallery_items:
        frame.destroy()
    gallery_items.clear()
    pred_text.set("Prediction: None")


def arrange_gallery():
    for i, frame in enumerate(gallery_items):
        row = i // 10
        col = i % 10
        frame.grid(row=row, column=col, padx=8, pady=8, sticky="nsew")
    for col in range(10):
        gallery_frame.grid_columnconfigure(col, weight=1)


def open_zoom_window(file_path):
    zoom_win = Toplevel(root)
    zoom_win.title("🔎 Zoomed Image")
    zoom_win.geometry(f"{screen_width//2}x{screen_height//2}")
    zoom_win.configure(bg="#101010")

    img = Image.open(file_path)
    zoom_win.img = img
    zoom_win.scale = 1.0
    display_image_in_zoom(zoom_win, img)

    btn_frame = ctk.CTkFrame(zoom_win, corner_radius=12, fg_color="#1c1c1c")
    btn_frame.pack(pady=12)

    ctk.CTkButton(btn_frame, text="➕ Zoom In",
                  command=lambda: zoom_image(zoom_win, 1.2),
                  fg_color="#007aff", hover_color="#0051a3").pack(side="left", padx=10)
    ctk.CTkButton(btn_frame, text="➖ Zoom Out",
                  command=lambda: zoom_image(zoom_win, 0.8),
                  fg_color="#007aff", hover_color="#0051a3").pack(side="left", padx=10)
    ctk.CTkButton(btn_frame, text="❌ Close",
                  command=zoom_win.destroy,
                  fg_color="#ff3b30", hover_color="#cc2e25").pack(side="left", padx=10)


def display_image_in_zoom(zoom_win, img):
    w, h = img.size
    new_w = int(w * zoom_win.scale)
    new_h = int(h * zoom_win.scale)
    resized = img.resize((new_w, new_h))
    tk_img = CTkImage(light_image=resized, size=(new_w, new_h))  # ✅ fixed

    if hasattr(zoom_win, "img_label"):
        zoom_win.img_label.configure(image=tk_img)
        zoom_win.img_label.image = tk_img
    else:
        zoom_win.img_label = ctk.CTkLabel(zoom_win, image=tk_img, text="")
        zoom_win.img_label.image = tk_img
        zoom_win.img_label.pack(expand=True)


def zoom_image(zoom_win, factor):
    zoom_win.scale *= factor
    display_image_in_zoom(zoom_win, zoom_win.img)


# ------------------------------
# Prediction function
# ------------------------------


def predict_images():
    
    if not gallery_items:
        pred_text.set("Prediction: No images")
        return

    # Create prediction window
    pred_win = Toplevel(root)
    pred_win.title("🧠 Prediction Results")
    pred_win.geometry(f"{screen_width//2}x{screen_height//2}")
    pred_win.configure(bg="#101010")       
    # ===============================
    # Collect all images
    # ===============================
    all_imgs = [np.array(card.img.convert("L").resize((200, 200))) / 255.0 for card in gallery_items]
    all_imgs = np.array(all_imgs)   # shape = (N,200,200)
    print("✅ All images shape:", all_imgs.shape)
    
    def extract_hog_features2(images, pixels_per_cell=(16,16), cells_per_block=(2,2), orientations=9):
        feats = []
        for img in images:
            hog_feat = hog(img, 
                           orientations=orientations, 
                           pixels_per_cell=pixels_per_cell, 
                           cells_per_block=cells_per_block, 
                           block_norm='L2-Hys', 
                           visualize=False, 
                           feature_vector=True)
            feats.append(hog_feat)
        return np.array(feats, dtype="float32")
    
    def extract_hog_features3(images, 
                                 target_size=(128,128), 
                                 pixels_per_cell=(8,8), 
                                 cells_per_block=(2,2), 
                                 orientations=9):
       feats = []
       for img in images:
           if len(img.shape) == 3:
               img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
           img_resized = cv2.resize(img, target_size)   
           hog_feat = hog(img_resized,
                           orientations=orientations,
                           pixels_per_cell=pixels_per_cell,
                           cells_per_block=cells_per_block,
                           block_norm='L2-Hys',
                           visualize=False,
                           feature_vector=True)
           feats.append(hog_feat)
       return np.array(feats, dtype="float32")

    
    def extract_hog_features4(images, pixels_per_cell=(8,8), cells_per_block=(2,2), orientations=9):
        all_feats = []
        for img in images:
            img = img.astype(np.float32)
            hog_feat = hog(img,
                           orientations=orientations,
                           pixels_per_cell=pixels_per_cell,
                           cells_per_block=cells_per_block,
                           block_norm='L2-Hys',
                           visualize=False,
                           feature_vector=True)
            all_feats.append(hog_feat)
        return np.array(all_feats, dtype=np.float32)

    def extract_wavelet_features5(images, wavelet="db1", level=2):
        all_feats = []
        for img in images:
            img = img.astype(np.float32)
            coeffs = pywt.wavedec2(img, wavelet=wavelet, level=level)
            feats = []
            for c in coeffs:
                if isinstance(c, tuple):
                   for arr in c:
                       feats.extend(arr.ravel())
                else:
                    feats.extend(c.ravel())
            all_feats.append(np.array(feats, dtype=np.float32))
        return np.array(all_feats, dtype=np.float32)

    def extract_hog_features5(images, pixels_per_cell=(16,16), cells_per_block=(2,2), orientations=9):
        all_feats = []
        for img in images:
            img = img.astype(np.float32)
            hog_feat = hog(img,
                           orientations=orientations,
                           pixels_per_cell=pixels_per_cell,
                           cells_per_block=cells_per_block,
                           block_norm='L2-Hys',
                           visualize=False,
                           feature_vector=True)
            all_feats.append(hog_feat)
        return np.array(all_feats, dtype=np.float32)
 
    def extract_combined_features5(images):
        wavelet_feats = extract_wavelet_features5(images)
        hog_feats     = extract_hog_features5(images)
        combined_feats = np.hstack([wavelet_feats, hog_feats])
        return combined_feats

    # CNN
    cnn_input = all_imgs.reshape(-1, 200, 200, 1)
    pp1 = (cnn.predict(cnn_input) > 0.5).astype(int).flatten()

    # Random Forest
    rf_ext = rf_ext = extract_hog_features2(all_imgs)
    scal_rf=joblib.load("scaler_rf_model.pkl")
    scal_rf_ext=scal_rf.transform(rf_ext)
    pp2 = rf.predict(scal_rf_ext)

    # SVM
    svm_ext = extract_hog_features3(all_imgs)
    pp3 = svm.predict(svm_ext)

    # KNN
    knn_ext = extract_hog_features4(all_imgs)
    scal_knn=joblib.load("scaler_knn_model.pkl")
    scal_ext_knn=scal_knn.transform(knn_ext)
    pp4 = knn.predict(scal_ext_knn)

    # Logistic Regression
    LR_ext = extract_combined_features5(all_imgs)
    pp5 = LogR.predict(LR_ext)

    # ===============================
    # Ensemble (majority vote)
    # ===============================
    votes = pp1 + pp2 + pp3 + pp4 + pp5
    Ensemble_prediction = (votes >= 3).astype(int)
    print("cnn", pp1)
    print("rf", pp2)
    print("svm", pp3)
    print("knn", pp4)
    print("lr", pp5)

    # ===============================
    # Show results
    # ===============================
    for i, (card, pred) in enumerate(zip(gallery_items, Ensemble_prediction)):
        pred_label_text = "Tumor Detected" if pred == 1 else "No Tumor"

        tk_img = CTkImage(light_image=card.img.resize((150, 150)), size=(150, 150))
        img_label = ctk.CTkLabel(pred_win, image=tk_img, text="")
        img_label.image = tk_img
        img_label.grid(row=i // 4, column=(i % 4) * 2, padx=10, pady=10)

        pred_label = ctk.CTkLabel(pred_win, text=pred_label_text,
                                  font=("Arial", 16), text_color="#00ff00")
        pred_label.grid(row=i // 4, column=(i % 4) * 2 + 1, padx=10, pady=10)

    pred_text.set("Prediction: Done ✅")



# ------------------------------
# Main Window
# ------------------------------
ctk.set_appearance_mode("dark")
ctk.set_default_color_theme("blue")

root = ctk.CTk()
root.title("✨ IDENTIFICATION OF BRAIN TUMOR ✨")

screen_width = root.winfo_screenwidth()
screen_height = root.winfo_screenheight()
root.geometry(f"{screen_width}x{screen_height}")

title_label = ctk.CTkLabel(root, text="🌌 IDENTIFICATION OF BRAIN TUMOR",
                           font=("Arial Rounded MT Bold", 34, "bold"),
                           text_color="#00c6ff")
title_label.pack(pady=20)

button_frame = ctk.CTkFrame(root, corner_radius=15, fg_color="#202020")
button_frame.pack(pady=10)

ctk.CTkButton(button_frame, text="📂 Select Images",
              command=select_images, font=("Arial", 18),
              fg_color="#007aff", hover_color="#0051a3").pack(side="left", padx=15, pady=10)

ctk.CTkButton(button_frame, text="🗑 Remove All",
              command=remove_all_images, font=("Arial", 18),
              fg_color="#ff3b30", hover_color="#cc2e25").pack(side="left", padx=15, pady=10)

pred_text = ctk.StringVar(value="Prediction: None")
ctk.CTkButton(button_frame, text="🧠 Predicted All",
              command=predict_images, font=("Arial", 18),
              fg_color="#00c6ff", hover_color="#0072ff").pack(side="left", padx=15, pady=10)

ctk.CTkLabel(button_frame, textvariable=pred_text,
             font=("Arial", 18), text_color="#00ff00").pack(side="left", padx=15, pady=10)

gallery_container = ctk.CTkScrollableFrame(root, orientation="vertical", height=screen_height - 200)
gallery_container.pack(fill="both", expand=True, pady=20)
gallery_frame = gallery_container

root.mainloop()


✅ All images shape: (4, 200, 200)
1/1 [==============================] - 0s 116ms/step
cnn [1 1 1 1]
rf [1 1 1 1]
svm [1 1 1 0]
knn [1 1 1 0]
lr [1 1 1 1]
